In [36]:
:dep ureq = { version = "2", features = ["json"] }
:dep serde_json = "1"
:dep urlencoding = "2"

In [44]:
// 問題文の #!/usr/bin/python をヒントにそのまま検索
let search_url =
    "https://en.wikipedia.org/w/api.php?action=query&list=search\
     &srsearch=%23!+python+interpreter&srlimit=5&format=json";

let json: serde_json::Value = ureq::get(search_url)
    .set("User-Agent", "Mozilla/5.0")
    .call().unwrap()
    .into_json().unwrap();

println!("=== 検索結果タイトル ===");
let mut top_title = String::new();
if let Some(results) = json["query"]["search"].as_array() {
    for (i, r) in results.iter().enumerate() {
        let title = r["title"].as_str().unwrap_or("");
        println!("  [{}] {}", i + 1, title);
        if i == 0 { top_title = title.to_string(); }
    }
}

// 最上位記事の本文を取得
println!("\n最上位記事: {}", top_title);
let ext_url = format!(
    "https://en.wikipedia.org/w/api.php?action=query&prop=extracts&exintro&explaintext&titles={}&format=json",
    urlencoding::encode(&top_title)
);

let j2: serde_json::Value = ureq::get(&ext_url)
    .set("User-Agent", "Mozilla/5.0")
    .call().unwrap()
    .into_json().unwrap();

let extract = j2["query"]["pages"]
    .as_object()
    .and_then(|p| p.values().next())
    .and_then(|p| p["extract"].as_str().map(|s| s.to_string()))
    .unwrap_or_default();

println!("\n=== 記事本文冒頭 ===\n{}\n", &extract[..extract.len().min(500)]);

// 7文字の英単語を頻度順に抽出
let mut freq: std::collections::HashMap<String, u32> = std::collections::HashMap::new();
for word in extract.split(|c: char| !c.is_alphabetic()) {
    if word.len() == 7 {
        *freq.entry(word.to_lowercase()).or_insert(0) += 1;
    }
}

let mut ranked: Vec<(String, u32)> = freq.into_iter().collect();
ranked.sort_by(|a, b| b.1.cmp(&a.1).then(a.0.cmp(&b.0)));

println!("=== 7文字英単語 (頻度順) ===");
for (i, (word, count)) in ranked.iter().take(20).enumerate() {
    println!("  #{:2}: {:12} ({}回)", i + 1, word, count);
}
if let Some((word, _)) = ranked.first() {
    println!("\n=> 最有力候補: \"{}\"", word.to_uppercase());
}

=== 検索結果タイトル ===
  [1] Python (programming language)
  [2] Stackless Python
  [3] Zen of Python
  [4] PyPy
  [5] Shebang (Unix)

最上位記事: Python (programming language)

=== 記事本文冒頭 ===
Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation, "plain English" naming, an extensive ("batteries-included") standard library, and garbage collection. Python supports multiple programming paradigms but with an emphasis on object-oriented programming and dynamic typing.
Guido van Rossum began working on Python in the late 1980s as a successor to the ABC programming language. Pyth

=== 7文字英単語 (頻度順) ===
  # 1: earlier      (2回)
  # 2: release      (2回)
  # 3: dynamic      (1回)
  # 4: english      (1回)
  # 5: garbage      (1回)
  # 6: general      (1回)
  # 7: library      (1回)
  # 8: machine      (1回)
  # 9: october      (1回)
  #10: popular      (1回)
  #11: project      (1回)
  #12: purpose      

()

# ksnctf Problem 10 (#!) 解析レポート

## 1. 概要
ksnctf Problem 10に対し、問題文の `#!/usr/bin/python` をヒントに、Rustを用いてFLAGの候補を抽出する自動化プログラムを作成・実行した。

## 2. 実装アプローチ
Jupyter Notebook環境にて、以下のステップでRustコードを実行。

1. キーワード検索: #! python interpreter をクエリとしてWikipedia APIを叩き、関連性の高い記事を抽出。

2. 本文の解析: 検索結果1位の記事（Python (programming language)）のテキストデータを取得。

3. 統計処理: FLAGの形式を想定し、本文中から7文字の英単語を抽出して出現頻度をカウント。

## 3. 実行結果と課題
### 実行データ
#### 検索結果上位:

Python (programming language)

Stackless Python

Zen of Python

PyPy

Shebang (Unix)

#### 抽出された7文字単語（頻度順）:

1位: earlier (2回)

2位: release (2回)

3位: dynamic (1回)

...以下、english, garbage など

### 考察
プログラムが導き出した最有力候補は "EARLIER" となり、本来のFLAG（正解）とは見当違いの結果となった。この要因として以下の2点が挙げられる。

1. 出現頻度の不足:
解析対象とした本文内において、ターゲットとなる7文字単語の出現回数が極めて低く、統計的な優位性を確保できなかった。

2. 抽出対象の不一致:
プログラムでは「検索結果1位の本文」を詳細に解析したが、実際には検索結果のタイトル一覧に正解が含まれていた。

## 4. 実行結果と課題
今回のケースでは、高度な本文解析プログラムを組むよりも、検索結果のメタデータ（タイトル）を俯瞰的に観察することが、最短ルートでFLAG（SHEBANG）に到達する鍵であった。

「自動化による抽出」と「人間によるパターンの気づき」のバランスがCTF攻略における重要な教訓となった。